### Logistic Regression from Scratch

This notebook demonstrates a **Logistic Regression** classifier implemented from scratch using NumPy and compares it directly with Scikit-Learn's standard implementation on the Breast Cancer dataset.

#### Key Mathematical Concepts

1. **The Sigmoid Activation Function**
   Logistic regression predicts the probability $p$ that a given input $X$ belongs to the positive class ($1$). We map a linear combination of input features ($z = Xw + b$) to a probability range between $0$ and $1$ using the **Sigmoid function**:
   $$\sigma(z) = \frac{1}{1 + e^{-z}}$$

2. **Binary Cross-Entropy Loss (Log Loss)**
   To train our model, we measure how far our predicted probabilities $y_{pred}$ are from the actual binary targets $y_{true}$ using the average log loss:
   $$\text{Loss} = -\frac{1}{N} \sum_{i=1}^{N} \left[ y_i \log(p_i) + (1 - y_i) \log(1 - p_i) \right]$$
   *Note: We clip the predicted probabilities near 0 and 1 (with a small epsilon) to avoid taking the logarithm of zero, which would result in mathematical errors (`NaN` or `-inf`).*

3. **Gradient Descent Optimization**
   We iteratively update the model weights ($w$) and bias ($b$) to minimize the loss. The gradients (partial derivatives of the loss function with respect to $w$ and $b$) are:
   $$\frac{\partial L}{\partial w} = \frac{1}{N} X^T (y_{pred} - y_{true})$$
   $$\frac{\partial L}{\partial b} = \frac{1}{N} \sum (y_{pred} - y_{true})$$
   
   During each epoch, we step in the direction of steepest descent scaled by our learning rate ($\alpha$):
   $$w \leftarrow w - \alpha \frac{\partial L}{\partial w}$$
   $$b \leftarrow b - \alpha \frac{\partial L}{\partial b}$$

#### Comparing Scratch vs. Scikit-Learn
To ensure a mathematically identical environment for comparison, we initialize `LogisticRegression(penalty=None)` in Scikit-Learn. This disables default L2 regularization (weight penalty) so both models seek to minimize the pure, unconstrained log loss function on standardized data.

In [51]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression as SklearnLogR
from sklearn.metrics import log_loss, accuracy_score

class ScratchLogisticRegression:
    def __init__(self, learning_rate: float = 0.1, epochs: int = 10000) -> None:
        self.lr = learning_rate
        self.epochs = epochs
        self.w = None
        self.b = 0.0

    def sigmoid(self, z: np.ndarray) -> np.ndarray:
        # Implement the Sigmoid formula
        y_pred = 1 / (1 + np.exp(-z))

        return y_pred

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        # Calculate the linear output
        z = np.dot(X, self.w) + self.b

        # Pass it through self.sigmoid()

        return self.sigmoid(z)

    def binary_cross_entropy_loss(self, y_true: np.ndarray, y_pred: np.ndarray) -> float:
        # We clip predictions to prevent log(0) which causes math errors
        epsilon = 1e-9
        y_pred = np.clip(y_pred, epsilon, 1 - epsilon)

        # Implement the Log Loss formula
        loss = (np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))) * -1

        return loss

    def fit(self, X: np.ndarray, y_true: np.ndarray) -> None:
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features)

        for epoch in range(self.epochs+1):
          y_pred = self.predict_proba(X)

          # Calculate gradients
          dw = (1 / n_samples) * np.dot(X.T, (y_pred - y_true))
          db = np.mean(y_pred - y_true)

          self.w = self.w - (self.lr * dw)
          self.b = self.b - (self.lr * db)

    def predict_classes(self, X: np.ndarray, threshold: float = 0.5) -> np.ndarray:
        # Convert probabilities into 1 or 0 classifications
        probabilities = self.predict_proba(X)
        return (probabilities >= threshold).astype(int)

def train_and_evaluate(features_list):
    # Fetch and Preprocess
    data = load_breast_cancer(as_frame=True)
    df = data.frame
    X_raw = df[features_list].values
    y = df['target'].values

    X_mean = np.mean(X_raw, axis=0)
    X_std = np.std(X_raw, axis=0)
    X_scaled = (X_raw - X_mean) / X_std

    # Train Scratch Model
    scratch_model = ScratchLogisticRegression(learning_rate=0.1, epochs=10000)
    scratch_model.fit(X_scaled, y)
    scratch_probs = scratch_model.predict_proba(X_scaled)
    scratch_classes = scratch_model.predict_classes(X_scaled)
    scratch_loss = scratch_model.binary_cross_entropy_loss(y, scratch_probs)
    scratch_acc = accuracy_score(y, scratch_classes)

    # Train Scikit-Learn Model
    sklearn_model = SklearnLogR(penalty=None, solver="lbfgs", max_iter=10000, tol=1e-10)
    sklearn_model.fit(X_scaled, y)
    sklearn_probs = sklearn_model.predict_proba(X_scaled)[:, 1]
    sklearn_classes = sklearn_model.predict(X_scaled)
    sklearn_loss = log_loss(y, sklearn_probs)
    sklearn_acc = accuracy_score(y, sklearn_classes)

    return {
        'scratch_acc': scratch_acc,
        'sklearn_acc': sklearn_acc,
        'scratch_loss': scratch_loss,
        'sklearn_loss': sklearn_loss,
        'scratch_weights': scratch_model.w,
        'sklearn_weights': sklearn_model.coef_[0]
    }

if __name__ == '__main__':
    features_2d = ['mean radius', 'mean texture']
    features_4d = ['mean radius', 'mean texture', 'mean perimeter', 'mean area']

    print("Running training evaluations...")
    results_2d = train_and_evaluate(features_2d)
    results_4d = train_and_evaluate(features_4d)

    print("\n========================== RESULTS COMPARISON ==========================")
    print(f"{'Metric':<15} | {'2 Features (Scratch)':<20} | {'2 Features (Sklearn)':<20} || {'4 Features (Scratch)':<20} | {'4 Features (Sklearn)':<20}")
    print("-" * 105)
    print(f"{'Accuracy':<15} | {results_2d['scratch_acc']*100:<19.2f}% | {results_2d['sklearn_acc']*100:<19.2f}% || {results_4d['scratch_acc']*100:<19.2f}% | {results_4d['sklearn_acc']*100:<19.2f}%")
    print(f"{'Log Loss':<15} | {results_2d['scratch_loss']:<20.6f} | {results_2d['sklearn_loss']:<20.6f} || {results_4d['scratch_loss']:<20.6f} | {results_4d['sklearn_loss']:<20.6f}")

    print("\n========================== FEATURE WEIGHTS ==========================")
    # Display weights for 2 features
    print("--- 2-Feature Model Weights ---")
    for i, feature in enumerate(features_2d):
        print(f"{feature:<15} | Scratch: {results_2d['scratch_weights'][i]:<10.4f} | Scikit-Learn: {results_2d['sklearn_weights'][i]:<10.4f}")

    # Display weights for 4 features
    print("\n--- 4-Feature Model Weights ---")
    for i, feature in enumerate(features_4d):
        print(f"{feature:<15} | Scratch: {results_4d['scratch_weights'][i]:<10.4f} | Scikit-Learn: {results_4d['sklearn_weights'][i]:<10.4f}")

Running training evaluations...

========================== RESULTS COMPARISON ==========================
Metric          | 2 Features (Scratch) | 2 Features (Sklearn) || 4 Features (Scratch) | 4 Features (Sklearn)
---------------------------------------------------------------------------------------------------------
Accuracy        | 89.10              % | 89.10              % || 89.63              % | 91.92              %
Log Loss        | 0.255820             | 0.255820             || 0.233614             | 0.185735            

========================== FEATURE WEIGHTS ==========================
--- 2-Feature Model Weights ---
mean radius     | Scratch: -3.7220    | Scikit-Learn: -3.7220   
mean texture    | Scratch: -0.9374    | Scikit-Learn: -0.9374   

--- 4-Feature Model Weights ---
mean radius     | Scratch: 0.6298     | Scikit-Learn: 33.1981   
mean texture    | Scratch: -0.9275    | Scikit-Learn: -1.0211   
mean perimeter  | Scratch: -4.3113    | Scikit-Learn: -27.9352  
